# 7 · Fundición: ¿se puede saber al colar si una pieza saldrá mal?

Una pieza se cuela, pasa ~47 minutos en el túnel de enfriamiento y solo entonces el escáner QC-01 dice si es
buena. Si se pudiera saber **al colar** qué moldes tienen riesgo, se podría actuar 47 minutos antes (retener
el molde, corregir el horno o la arena).

El jefe de calidad tiene sus reglas (las del reto de trazabilidad): colada ≥ 1405 °C, humedad ≤ 4,3 %, siempre
inoculada. En este cuaderno:

* comprobamos **cuánto aciertan** esas reglas;
* dejamos que un **árbol de decisión** aprenda sus propios límites **de los datos**;
* usamos la **probabilidad** de rechazo para elegir a partir de qué riesgo avisar.

Datos: el conjunto `fundicion` (una fila por pieza inspeccionada, con las condiciones de su molde).

In [ ]:
from labia import datos, guardar
import numpy as np, pandas as pd, matplotlib.pyplot as plt

df = datos("fundicion")
print(len(df), "piezas ·", df.rechazo.sum(), "rechazos", f"({100 * df.rechazo.mean():.1f} %)")
df.groupby("defecto").size().to_frame("piezas")

## La etiqueta engaña: el retraso del túnel

La columna `averia` es la avería activa **cuando se inspeccionó** la pieza. Mira la tasa de rechazo por tramo de
la campaña: los tramos «sin avería» que siguen a una avería también tienen rechazos.

In [ ]:
tramo = (df.averia != df.averia.shift()).cumsum()
t = df.groupby(tramo).agg(averia=("averia", "first"), piezas=("rechazo", "size"), rechazo_pct=("rechazo", "mean"))
t["rechazo_pct"] = (100 * t.rechazo_pct).round(1)
t

> Las piezas que se inspeccionan en un tramo «sin avería» se colaron durante la avería anterior. Por eso **no**
> entrenamos con la avería como etiqueta: usamos lo que le pasó a **cada molde** (su temperatura de colada, su
> humedad, si se inoculó) y el resultado de **su** pieza.

## ¿Dónde está el riesgo?

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (col, bordes) in zip(ejes[:2], [("t_colada", np.arange(1360, 1490, 10)), ("humedad", np.arange(3.2, 5.0, 0.2))]):
    g = df.groupby(pd.cut(df[col], bordes), observed=True).rechazo.agg(["mean", "size"])
    ax.bar(range(len(g)), 100 * g["mean"], color="tab:red", alpha=.7)
    ax.set_xticks(range(len(g)), [f"{i.left:g}" for i in g.index], rotation=45)
    ax.set_title(f"% de rechazo según {col}"); ax.grid(alpha=.3, axis="y")
g = df.groupby("inoculada").rechazo.mean().mul(100)
ejes[2].bar(["sin inocular", "inoculada"], [g.get(0, 0), g.get(1, 0)], color="tab:red", alpha=.7)
ejes[2].set_title("% de rechazo según inoculación"); ejes[2].grid(alpha=.3, axis="y")
plt.tight_layout();

El riesgo **no** salta de golpe en un límite: sube poco a poco (el proceso es aleatorio). Ninguna regla puede
acertar todas las piezas.

## Las reglas del jefe de calidad

Solo con lo que se sabe **al colar** (la temperatura de desmoldeo llega 47 minutos tarde).

In [ ]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix

def informe(nombre, real, aviso):
    print(f"{nombre:28s} avisa {aviso.sum():4d} moldes · precisión {100 * precision_score(real, aviso):5.1f} % "
          f"· detección {100 * recall_score(real, aviso):5.1f} %")

reglas = ((df.t_colada < 1405) | (df.humedad > 4.3) | (df.inoculada == 0)).astype(int)
informe("Reglas del jefe de calidad", df.rechazo, reglas)

* **Precisión**: de los moldes avisados, cuántos salieron mal (si es baja, retienes muchos moldes buenos).
* **Detección**: de las piezas rechazadas, cuántas se avisaron al colar.

## Un árbol de decisión aprende sus propios límites

Separamos las piezas en el **tiempo**: el primer 70 % para entrenar, el 30 % final para evaluar.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

ENTRADAS = ["t_colada", "humedad", "inoculada"]
corte = int(len(df) * 0.7)
ent, prueba = df.iloc[:corte], df.iloc[corte:]

arbol = DecisionTreeClassifier(max_depth=3, min_samples_leaf=10, random_state=0)
arbol.fit(ent[ENTRADAS], ent.rechazo)
print(export_text(arbol, feature_names=ENTRADAS, show_weights=True))

In [ ]:
plt.figure(figsize=(16, 6))
plot_tree(arbol, feature_names=ENTRADAS, class_names=["OK", "RECHAZO"], filled=True, proportion=True,
          rounded=True, fontsize=8);

Busca los **límites** que ha encontrado el árbol para `t_colada` y `humedad`. ¿Se parecen a los del jefe de
calidad? Nadie le ha dicho al árbol qué es una colada fría: lo ha sacado de los datos.

## Probabilidad y umbral

Cada hoja del árbol da una **probabilidad** de rechazo (la proporción de rechazos que vio en esa hoja). Avisar
o no depende del **umbral**: con un umbral bajo se avisa más (más detección, menos precisión).

In [ ]:
riesgo = arbol.predict_proba(prueba[ENTRADAS])[:, 1]
reglas_p = ((prueba.t_colada < 1405) | (prueba.humedad > 4.3) | (prueba.inoculada == 0)).astype(int)
informe("Reglas (prueba)", prueba.rechazo, reglas_p)
for umbral in (0.10, 0.15, 0.25, 0.50):
    informe(f"Árbol, umbral {umbral:.2f}", prueba.rechazo, (riesgo >= umbral).astype(int))

In [ ]:
from sklearn.metrics import precision_recall_curve
pr, rc, th = precision_recall_curve(prueba.rechazo, riesgo)
plt.figure(figsize=(6, 4.5))
plt.plot(100 * rc, 100 * pr, "o-", label="árbol (cada punto, un umbral)")
plt.plot(100 * recall_score(prueba.rechazo, reglas_p), 100 * precision_score(prueba.rechazo, reglas_p),
         "s", ms=10, color="tab:red", label="reglas del jefe de calidad")
plt.xlabel("detección (%)"); plt.ylabel("precisión (%)"); plt.grid(alpha=.3); plt.legend();

> **¿La IA gana a las reglas?** Si el punto rojo queda **sobre** la curva, las reglas ya son tan buenas como el
> árbol: el jefe de calidad conocía bien su proceso. Lo que aporta el árbol es que ha aprendido los límites
> **sin** él, y que da un riesgo graduado (no es lo mismo un 12 % que un 70 %).

Elige el umbral y publica el modelo. Node-RED lo usará **al colar** cada molde (variable
`Trazabilidad.UltimaColada`) y comprobará el acierto cuando su pieza llegue al escáner.

In [ ]:
UMBRAL = 0.15
aviso = (riesgo >= UMBRAL).astype(int)
arbol.fit(df[ENTRADAS], df.rechazo)          # para publicar, con todas las piezas
guardar(arbol, "fundicion-riesgo", entradas=ENTRADAS, tipo="clasificacion", salida="rechazo",
        descripcion="Árbol de decisión (profundidad 3): riesgo de rechazo con lo que se sabe al colar",
        metricas={"umbral": UMBRAL,
                  "precision_pct": round(100 * float(precision_score(prueba.rechazo, aviso)), 1),
                  "deteccion_pct": round(100 * float(recall_score(prueba.rechazo, aviso)), 1)})

## ✏️ Tu turno

1. Cambia `max_depth` a 1, 2 y 6. ¿Qué límites encuentra con profundidad 1? ¿Mejora con 6 o empieza a aprender
   ruido (mira los límites que salen)?
2. Añade `t_desmoldeo` a las entradas. ¿Mejora? ¿Podrías usarlo **al colar**?
3. Prueba `RandomForestClassifier` y `LogisticRegression` (con `predict_proba`). ¿Mejoran al árbol? ¿Cuál
   explicarías mejor a un operador?
4. El molde rechazado cuesta 100 € de material; retener un molde para revisarlo cuesta 8 €. ¿Qué umbral
   elegirías? (Pista: calcula el coste total para cada umbral.)
5. ¿Qué avería de la campaña no se puede ver al colar? (Pista: la avería 4 del escáner, o la del desmoldeo.)